# Simulating the Linear Encoding Quantum Lattice Gas Automata

The LQLGA encoding assigns one qubit to every (gridpoint, velocity) pair, so the register width
grows linearly with the domain rather than logarithmically — 7 gridpoints of D1Q3 already need
21 qubits.

`qlbm` builds its circuits as `qarp` blocks and simulates them with `qarp`'s statevector simulator.
A run is described by a `SimulationConfig`, which ties the four algorithmic circuits — initial
conditions, time step algorithm, postprocessing, and measurement — to a compiler optimization level
and a shot budget. `cfg.validate()` type-checks the configuration and `cfg.prepare_for_simulation()`
lowers every component into a built `qarp` block. A `QarpRunner` then drives the time loop and writes
one ParaView frame per time step into `output_dir/paraview`.

`statevector_snapshots=True` is the fast path: the runner hands the state produced at step $k$ back to
the simulator as the *initial state* of step $k+1$, so simulating $n$ steps costs $n$ applications of
the algorithm instead of $\frac{n(n+1)}{2}$. Passing `qarp.EXACT` instead of a shot count returns exact
outcome probabilities rather than sampled counts.

> **Demo-sized.** The lattice below is a 7-point D1Q3 domain with one bounce-back obstacle,
> simulated for 10 time steps. Simulation cost grows exponentially in the number of qubits, so scale the lattice and
> `NUM_STEPS` up deliberately.

In [ ]:
from qlbm.components import EmptyPrimitive
from qlbm.components.lqlga import (
    LQLGA,
    LQGLAInitialConditions,
    LQLGAGridVelocityMeasurement,
)
from qlbm.infra import QarpRunner, SimulationConfig
from qlbm.lattice import LQLGALattice
from qlbm.tools.utils import create_directory_and_parents

In [ ]:
lattice = LQLGALattice(
    {
        "lattice": {
            "dim": {"x": 7},
            "velocities": "D1Q3",
        },
        "geometry": [{"shape": "cuboid", "x": [3, 5], "boundary": "bounceback"}],
    },
)


output_dir = "qlbm-output/lqlga-d1q3-7-qarp"
create_directory_and_parents(output_dir)

In [ ]:
# Number of shots to sample for each time step
NUM_SHOTS = 2**10

# Number of time steps to simulate
NUM_STEPS = 10

In [ ]:
cfg = SimulationConfig(
    initial_conditions=LQGLAInitialConditions(
        lattice, [(tuple([2]), (True, True, True))]
    ),
    algorithm=LQLGA(lattice),
    postprocessing=EmptyPrimitive(lattice),
    measurement=LQLGAGridVelocityMeasurement(lattice),
    optimization_level=0,
    shots=NUM_SHOTS,
)

cfg.validate()
cfg.prepare_for_simulation()

In [ ]:
# Create a runner object to simulate the circuit
runner = QarpRunner(cfg, lattice, seed=1234)


# Snapshots carry the statevector between time steps, making the loop O(NUM_STEPS)
runner.run(
    NUM_STEPS,  # Number of time steps
    NUM_SHOTS,  # Number of shots per time step
    output_dir,
    statevector_snapshots=True,
)